# VLM-DENTAL — Stage 1: Supervised Fine-Tuning (SFT)

This notebook trains **Qwen/Qwen3.5-9B** using native **BF16 LoRA** ($r=32, \alpha=64$) on verified expert dental clinical traces across 3 staged curriculum milestones.

### Core Architectural & Clinical Invariants:
- **3-Stage Curriculum with Negative Controls Calibration**:
  - **Stage 1a (`dentex_alone`)**: 678 DENTEX disease traces + 27 healthy DENTEX negative controls $\rightarrow$ `qwen3_5_9b_sft_{track}_dentex`.
  - **Stage 1b (`dentex_tufts_overlap`)**: DENTEX + Tufts overlapping disease (caries, periapical) + Healthy controls (DENTEX + Tufts) $\rightarrow$ `qwen3_5_9b_sft_{track}_dentex_tufts_overlap`.
  - **Stage 1c (`multicohort_all`)**: DENTEX (4 findings) + Tufts Full (all 4 findings on their own) + Full healthy controls $\rightarrow$ `qwen3_5_9b_sft_{track}_multicohort_all`.
- **Multimodal Vision Projector LoRA**: Adapts `merger.mlp.0` and `merger.mlp.2` alongside LLM attention/MLP projections, specializing radiographic feature projection without catastrophic forgetting of base ViT representations.
- **Native Image Resolutions**: Native $2:1$ panoramic X-ray aspect ratios and resolutions preserved without downsampling, leveraging the 128 GB TPU v5e-8 HBM.
- **Hardware Optimization**: Native 8-way FSDPv2 on **Google Cloud TPU v5e-8** and multi-GPU Accelerate.
- **Strict Track Segregation**:
  - **Track A (`with_tools`)**: Multi-turn agent trained on real workstation tool-use traces (8 tools).
  - **Track B (`no_tools`)**: Single-turn direct radiologist trained on tool-free Chain-of-Thought (CoT) traces.
- **Cosine Warmup & Gradient Clipping**: 5% linear warmup, cosine decay to $1\times 10^{-6}$, and gradient norm clipping at $1.0$.
- **Validation Checkpointing**: 5% held-out validation split with `best_adapter` retention.
- **Hugging Face Hub Checkpoint Sync**: Checkpoints (~760 MB LoRA + optimizer) are pushed every 25 steps to survive Kaggle 9-hour session limits and enable seamless multi-account resume.

## 1. Platform Detection & Shallow Repository Clone

Detects runtime environment (Kaggle vs Colab vs Local) and performs a shallow clone (`--depth 1`) to eliminate history download overhead.

In [ ]:
import os
import sys
from pathlib import Path

# 1. Detect platform environment
IS_KAGGLE = os.path.exists("/kaggle") or "KAGGLE_KERNEL_RUN_TYPE" in os.environ
IS_COLAB = "google.colab" in sys.modules or "COLAB_GPU" in os.environ
PLATFORM_NAME = "Kaggle" if IS_KAGGLE else ("Colab" if IS_COLAB else "Local PC / Server")
print(f"[PLATFORM] Detected Runtime Environment: {PLATFORM_NAME}")

# 2. Shallow clone repository (--depth 1) to conserve bandwidth, disk space, and time
REPO_NAME = "VLM-DENTAL"
REPO_URL = "https://github.com/rezaxr14/VLM-DENTAL.git"

if not os.path.exists(REPO_NAME) and not os.path.exists("dental_agent"):
    print(f"[CLONE] Performing shallow clone (--depth 1) of {REPO_URL}...")
    !git clone --depth 1 {REPO_URL}
    %cd {REPO_NAME}
elif os.path.exists(REPO_NAME):
    %cd {REPO_NAME}
    print(f"[WORKSPACE] Switched directory to {os.getcwd()}")
else:
    print(f"[WORKSPACE] Already inside repository root: {os.getcwd()}")

## 2. Hardened Dependency Installation & Environment Setup

Installs core project dependencies before calling any hardware-specific libraries. Configures `PJRT_DEVICE=TPU` for Kaggle TPU v5e-8.

In [ ]:
# Configure PJRT device on TPU platforms before importing torch/xla
if IS_KAGGLE:
    os.environ["PJRT_DEVICE"] = "TPU"

# Install VLM-DENTAL in editable mode and essential PEFT / training packages
# Note: vLLM is completely purged to avoid CUDA binary incompatibilities on TPU
!pip install -q -e .
!pip install -q peft trl datasets accelerate huggingface_hub ultralytics python-dotenv qwen-vl-utils tabulate

## 3. Hardware & Cloud TPU v5e-8 Topology Detection

Initializes PyTorch/XLA on TPU v5e-8 (reporting 8-way FSDPv2 device count and chip topology) or reports CUDA GPUs.

In [ ]:
import torch

IS_TPU = False
DEVICE_STR = "cpu"

try:
    import torch_xla.core.xla_model as xm
    device = xm.xla_device()
    IS_TPU = True
    hw_name = xm.xla_device_hw(device)
    world_size = xm.xrt_world_size()
    DEVICE_STR = f"Cloud TPU ({hw_name}) - World Size: {world_size} chips"
    print(f"[HARDWARE] SUCCESS: Detected {DEVICE_STR}")
    print(f"[HARDWARE] 8-Way FSDPv2 SPMD ready across 128 GB total HBM.")
except Exception as e:
    if torch.cuda.is_available():
        gpu_count = torch.cuda.device_count()
        gpu_name = torch.cuda.get_device_name(0)
        DEVICE_STR = f"CUDA ({gpu_count}x {gpu_name})"
        print(f"[HARDWARE] SUCCESS: Detected {DEVICE_STR}")
    else:
        print(f"[HARDWARE WARNING] Running on CPU (Testing only): {e}")

print(f"[FRAMEWORK] PyTorch: {torch.__version__}")

## 4. Secrets Diagnostics, Hub Auth & Verified Traces Sync

Audits environment variables from `.env`, Kaggle Secrets Vault, and Colab Userdata, authenticates with Hugging Face Hub, and synchronizes verified clinical trace splits.

In [ ]:
from dotenv import load_dotenv
from huggingface_hub import login
from tabulate import tabulate

# 1. Load local .env if present
load_dotenv()

# Helper for Kaggle Secrets / Colab Userdata ingestion
def get_secret_multisource(key: str, default: str | None = None) -> tuple[str | None, str]:
    val = os.environ.get(key)
    if val:
        return val, ".env / OS Env"

    if IS_KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            sec = UserSecretsClient().get_secret(key)
            if sec:
                os.environ[key] = sec
                return sec, "Kaggle Secrets"
        except Exception:
            pass

    if IS_COLAB:
        try:
            from google.colab import userdata
            sec = userdata.get(key)
            if sec:
                os.environ[key] = sec
                return sec, "Colab Userdata"
        except Exception:
            pass

    if default is not None:
        os.environ[key] = default
        return default, "Default Fallback"

    return None, "Not Set"

def mask_secret(val: str | None, is_secret: bool = True) -> str:
    if not val:
        return "---"
    if not is_secret:
        return val
    if len(val) <= 8:
        return "********"
    return f"{val[:4]}...{val[-4:]}"

# Audit list of project environment variables & tokens
TOKEN_SPECS = [
    ("HF_TOKEN", True, None),
    ("HF_ARTIFACT_REPO", False, "Reza-Nadimi/vlm-dental-models"),
    ("HF_TRACES_REPO", False, "Reza-Nadimi/vlm-dental-traces"),
    ("DENTEX_IMAGES_REPO", False, "Reza-Nadimi/dentex-images-panoramic"),
    ("TUFTS_IMAGES_REPO", False, "Reza-Nadimi/tufts-images-panoramic"),
    ("OPENAI_API_KEY", True, None),
    ("ANTHROPIC_API_KEY", True, None),
    ("GEMINI_API_KEY", True, None),
    ("GROQ_API_KEY", True, None),
    ("NVIDIA_API_KEY", True, None),
    ("WANDB_API_KEY", True, None),
    ("DENTAL_AGENT_DATA_DIR", False, "data"),
]

status_table = []
for key, is_sec, def_val in TOKEN_SPECS:
    val, source = get_secret_multisource(key, def_val)
    status_str = "[SET]" if val else "[NOT SET]"
    status_table.append([key, status_str, source, mask_secret(val, is_sec)])

print("=" * 80)
print("VLM-DENTAL: ENVIRONMENT & AUTHENTICATION DIAGNOSTICS")
print("=" * 80)
print(tabulate(status_table, headers=["Environment Variable", "Status", "Detection Source", "Configured / Masked Value"], tablefmt="fancy_grid"))

# 2. Authenticate Hugging Face Hub
hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    try:
        login(token=hf_token, add_to_git_credential=True)
        print("\n[AUTH] Successfully authenticated with Hugging Face Hub.")
    except Exception as e:
        print(f"\n[AUTH WARNING] Hugging Face login failed: {e}")
else:
    print("\n[AUTH WARNING] No HF_TOKEN detected. Checkpoint upload will be disabled unless logged in interactively:")
    login()

# 3. Synchronize verified clinical trace splits from Hugging Face
print("\n[SYNC] Synchronizing verified clinical trace splits from Hugging Face...")
!python scripts/sync_traces_hf.py --download

## 5. Interactive SFT Configuration & Curriculum Stage Selection

Select the experimental curriculum stage, training track, and hyperparameters. Displays a structured manifest before execution.

In [ ]:
from tabulate import tabulate

# =========================================================================
# STAGE 1 SFT EXECUTION PARAMETERS
# =========================================================================
# Curriculum Stage:
#   - 'dentex_alone'          : Stage 1a: DENTEX Alone + DENTEX Healthy Controls
#   - 'dentex_tufts_overlap'  : Stage 1b: DENTEX + Tufts Overlap + Negative Controls
#   - 'multicohort_all'       : Stage 1c: Full Multi-Cohort: DENTEX + Tufts All 4 Findings + Full Negative Controls
STAGE = "dentex_alone"

# Training Track:
#   - 'with_tools' : Multi-turn diagnostic agent with 8 workstation tools
#   - 'no_tools'   : Direct radiologist (tool-free Chain-of-Thought)
TRACK = "with_tools"

# Vision LoRA Mode: 'projector' (merger.mlp.0, merger.mlp.2) or 'none'
LORA_TARGET_VISION = "projector"

# Precision: 'bf16' (native on Cloud TPU v5e-8 and Ampere+ GPUs), 'fp16', or 'qlora'
PRECISION = "bf16"

# Hyperparameters
DATA_DIR = "data"
BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 16
EPOCHS = 3
LEARNING_RATE = 2e-5
WARMUP_RATIO = 0.05
MAX_GRAD_NORM = 1.0
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05
EVAL_EVERY_STEPS = 25

# Checkpoint Sync & Kaggle Continuity
HF_REPO = os.environ.get("HF_ARTIFACT_REPO", "Reza-Nadimi/vlm-dental-models")
PUSH_EVERY_STEPS = 25
RESUME = False

# Expected Checkpoint & Manifest Table
target_checkpoint = f"data/models/qwen3_5_9b_sft_{TRACK}_{STAGE}"
hf_target_folder = f"sft/qwen3_5_9b_sft_{TRACK}_{STAGE}"

manifest_data = [
    ["Curriculum Stage", STAGE],
    ["Training Track", TRACK],
    ["Base Model ID", "Qwen/Qwen3.5-9B"],
    ["Vision LoRA Adapter", f"{LORA_TARGET_VISION} (merger.mlp.0, merger.mlp.2)" if LORA_TARGET_VISION == "projector" else "none"],
    ["Native Resolutions", "Enabled (Zero pixel downsampling / clamping)"],
    ["Precision", PRECISION],
    ["Effective Batch Size", f"{BATCH_SIZE * GRAD_ACCUM_STEPS}"],
    ["Peak Learning Rate", f"{LEARNING_RATE} (Cosine with {int(WARMUP_RATIO*100)}% warmup)"],
    ["Gradient Clipping", f"max_norm = {MAX_GRAD_NORM}"],
    ["LoRA Config", f"r={LORA_R}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}"],
    ["Validation Eval", f"Every {EVAL_EVERY_STEPS} steps (best_adapter tracking)"],
    ["Target Local Checkpoint", target_checkpoint],
    ["HF Models Repository", HF_REPO if HF_REPO else "Disabled"],
    ["HF Checkpoint Subfolder", hf_target_folder],
    ["Resume from HF", str(RESUME)],
]

print("=" * 75)
print("VLM-DENTAL: STAGE 1 SFT EXECUTION MANIFEST")
print("=" * 75)
print(tabulate(manifest_data, headers=["Configuration Parameter", "Assigned Value"], tablefmt="fancy_grid"))

## 6. Launch SFT Training Pipeline

Constructs explicit CLI execution command with full hyperparameter passing and streams live training output.

In [ ]:
cmd = [
    "python", "scripts/train_sft.py",
    "--track", TRACK,
    "--stage", STAGE,
    "--data-dir", DATA_DIR,
    "--lora-target-vision", LORA_TARGET_VISION,
    "--precision", PRECISION,
    "--batch-size", str(BATCH_SIZE),
    "--gradient-accumulation-steps", str(GRAD_ACCUM_STEPS),
    "--learning-rate", str(LEARNING_RATE),
    "--warmup-ratio", str(WARMUP_RATIO),
    "--max-grad-norm", str(MAX_GRAD_NORM),
    "--epochs", str(EPOCHS),
    "--lora-r", str(LORA_R),
    "--lora-alpha", str(LORA_ALPHA),
    "--lora-dropout", str(LORA_DROPOUT),
    "--eval-every-steps", str(EVAL_EVERY_STEPS),
    "--push-every-steps", str(PUSH_EVERY_STEPS),
]

if HF_REPO:
    cmd.extend(["--hf-repo", HF_REPO])

if RESUME and HF_REPO:
    cmd.extend(["--resume-hf", HF_REPO])

cmd_str = " ".join(cmd)
print(f"[LAUNCHING PIPELINE] Executing:\n{cmd_str}\n")
!{cmd_str}

## 7. Training Loss & Convergence Visualizer

Plots conversational assistant training loss and held-out validation loss curves, reporting initial, best validation, and final loss metrics.

In [ ]:
import json
import matplotlib.pyplot as plt

output_dir = f"data/models/qwen3_5_9b_sft_{TRACK}_{STAGE}"
log_file = f"{output_dir}/training_loss.jsonl"

if os.path.exists(log_file):
    steps, train_losses, val_steps, val_losses = [], [], [], []
    with open(log_file, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rec = json.loads(line)
                step = rec.get("step", len(steps))
                steps.append(step)
                train_losses.append(rec.get("loss", 0.0))
                if "val_loss" in rec:
                    val_steps.append(step)
                    val_losses.append(rec["val_loss"])

    plt.figure(figsize=(11, 5))
    plt.plot(steps, train_losses, label="Assistant Training Loss", color="#1f77b4", lw=1.8, alpha=0.85)
    if val_losses:
        plt.plot(val_steps, val_losses, label="Validation Loss (5% held-out)", color="#d62728", marker="o", lw=2)

    plt.title(f"VLM-DENTAL Stage 1 SFT Convergence — Stage: {STAGE.upper()} | Track: {TRACK.upper()}")
    plt.xlabel("Optimization Steps")
    plt.ylabel("Assistant Conversational Cross-Entropy Loss")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

    best_val_str = f" | Best Val Loss: {min(val_losses):.4f}" if val_losses else ""
    print(f"[METRICS] Initial Loss: {train_losses[0]:.4f} -> Final Loss: {train_losses[-1]:.4f}{best_val_str}")
    print(f"[CHECKPOINTS] Checkpoints saved to {output_dir}")
else:
    print(f"[INFO] Log file {log_file} not found yet. Execute Cell 6 to run training.")